In [1]:
import geopandas as gpd
import pandas as pd
from shapely import wkt

In [16]:
df = gpd.read_file('/Users/brendanjarrell/Desktop/hitl_reviewDS.csv')

df["st_astext"] = df["st_astext"].apply(wkt.loads)
hitl_gdf = gpd.GeoDataFrame(
    df,
    geometry="st_astext",
    crs="EPSG:4326"  # adjust if coordinates are not lon/lat
)

hitl_gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

# Count number of geometries.

In [17]:
sample_multipolygon = hitl_gdf['st_astext'].iloc[0]
num_geometries = len(sample_multipolygon.geoms)
print(f"Number of geometries in the first MultiPolygon: {num_geometries}")

hitl_gdf['geometry_count'] = hitl_gdf['st_astext'].apply(lambda geom: len(geom.geoms))
print("\nFirst 5 rows with geometry counts:")
print(hitl_gdf[['st_astext']].head())

Number of geometries in the first MultiPolygon: 22

First 5 rows with geometry counts:
                                           st_astext
0  MULTIPOLYGON (((25.59746 74.29779, 25.59814 74...
1  MULTIPOLYGON (((26.11107 73.59329, 26.11176 73...
2  MULTIPOLYGON (((-14.3557 48.05351, -14.35501 4...
3  MULTIPOLYGON (((-91.66586 0.58359, -91.66105 0...
4  MULTIPOLYGON (((-91.35618 0.23134, -91.35412 0...


# Size of the largest and median most polygon in a multipolygon.

In [28]:
hitl_gdf_newProj = hitl_gdf.to_crs('EPSG:6933')

In [45]:
hitl_gdf_newProj.iloc[0]['st_astext'].area

10430189.37089026

In [ ]:
hitl_gdf_newProj['largest_area'] = hitl_gdf_newProj['st_astext'].apply(lambda geom: max(part.area for part in geom.geoms))


0       3.896792e+06
1       3.868119e+06
2       1.324534e+07
3       3.362978e+07
4       7.254290e+06
            ...     
7966    7.089307e+06
7967    3.315423e+06
7968    1.935697e+06
7969    3.294313e+06
7970    8.333337e+06
Name: largest_area, Length: 7971, dtype: float64

In [63]:
test = hitl_gdf_newProj['st_astext'].iloc[0]

max([part.area for part in test.geoms])

3896792.4023594037

# Median pixel area of a multipolygon.

In [73]:
test = hitl_gdf_newProj['st_astext'].iloc[0]
num_geom = len(test.geoms)
middle = num_geom // 2
areas = [part.area for part in test.geoms]
median = sorted(areas)[middle]

# Apply to all members of the dataframe
def median_area(multipoly):
    num_geom = len(multipoly.geoms)
    middle = num_geom // 2
    areas = [part.area for part in multipoly.geoms]
    return sorted(areas)[middle]

hitl_gdf_newProj['median_area'] = hitl_gdf_newProj['st_astext'].apply(median_area)


In [74]:
hitl_gdf_newProj.head(5)

,id,slick_timestamp,st_astext,machine_cls,hitl_cls,machine_confidence,length,area,perimeter,centroid,polsby_popper,fill_factor,centerlines,aspect_ratio_factor,max_source_collated_score,geometry_count,largest_area,median_area
0,3273057,2024-07-21 15:44:47,"MULTIPOLYGON (((2469803.506 7065900.626, 24698...",5,1,0.9600062966346741,90479.42156099,10430194.534472644,179240.77044128766,0101000020E61000001933401D625139403062AE07B3AA...,0.004079704134171181,0.17341434501669312,"{""type"": ""FeatureCollection"", ""features"": [{""i...",0.9998600689270918,-2.997074592798687,22,3.896792e+06,3129.929236
1,3278751,2024-07-21 15:44:35,"MULTIPOLYGON (((2519359.921 7040735.513, 25194...",5,1,0.945455014705658,187059.15992051,17981273.724743545,347689.96294476854,0101000020E6100000347A485AE2923940F5E74B91A395...,0.001869157978317394,0.08627849616058189,"{""type"": ""FeatureCollection"", ""features"": [{""i...",0.9999936640027093,-3.0911078359252775,55,3.868119e+06,1653.120907
2,3264093,2024-07-20 07:05:18,"MULTIPOLYGON (((-1385127.9 5449983.516, -13850...",5,1,0.9159122705459595,76377.44526714,22870602.19789487,213312.21003821932,0101000020E6100000AE579E00649A2DC0A7624577230D...,0.006316202329465258,0.45344901672487353,"{""type"": ""FeatureCollection"", ""features"": [{""i...",0.9999960041891336,-4.458064854763842,11,1.324534e+07,46877.610144
3,3030492,2024-08-10 11:50:22,"MULTIPOLYGON (((-8844497.568 74449.753, -88440...",5,1,0.9863162040710449,20201.06759414,34918265.7979765,151266.87753361906,0101000020E610000048A41BD082ED56C0F75646F00C07...,0.01917674291267693,0.19226207056489528,"{""type"": ""FeatureCollection"", ""features"": [{""i...",0.40790214755351906,-1.0409978140947282,35,3.362978e+07,5809.306902
4,3022415,2024-08-21 00:27:07,"MULTIPOLYGON (((-8814617.986 29513.364, -88144...",5,1,0.9152988195419312,11577.03241622,16737114.913490694,84535.91637279905,0101000020E610000032DFBEFF6BD756C028093ABC1562...,0.02943119745078223,0.36656557433898673,"{""type"": ""FeatureCollection"", ""features"": [{""i...",0.3093327878017077,NULL,10,7.254290e+06,23220.761310


In [77]:
# Exporting modified dataframe to CSV.
hitl_gdf_newProj_red = hitl_gdf_newProj.drop(columns=['hitl_cls','machine_cls','centroid','centerlines', 'st_astext', 'max_source_collated_score', 'slick_timestamp', 'machine_confidence'], axis=1)
hitl_gdf_newProj_red.to_csv('/Users/brendanjarrell/Desktop/hitl_featsAdded_011326.csv', index=False)

# Doing Ethan's thing

In [36]:
hitl_gdf = gpd.read_file('/Users/brendanjarrell/Desktop/hitl_reviewDS.csv')
hitl_gdf = hitl_gdf[hitl_gdf['hitl_cls'].isin(['1','6','7','8'])]
hitl_gdf["is_slick"] = hitl_gdf['hitl_cls'].isin(['6','7','8'])


In [38]:
hitl_gdf.head()

,id,slick_timestamp,st_astext,machine_cls,hitl_cls,machine_confidence,length,area,perimeter,centroid,polsby_popper,fill_factor,centerlines,aspect_ratio_factor,max_source_collated_score,is_slick
0,3273057,2024-07-21 15:44:47,"MULTIPOLYGON(((25.597458 74.297791,25.598145 7...",5,1,0.9600062966346741,90479.42156099,10430194.534472644,179240.77044128766,0101000020E61000001933401D625139403062AE07B3AA...,0.004079704134171181,0.17341434501669312,"{""type"": ""FeatureCollection"", ""features"": [{""i...",0.9998600689270918,-2.997074592798687,False
1,3278751,2024-07-21 15:44:35,"MULTIPOLYGON(((26.111069 73.593292,26.111755 7...",5,1,0.945455014705658,187059.15992051,17981273.724743545,347689.96294476854,0101000020E6100000347A485AE2923940F5E74B91A395...,0.001869157978317394,0.08627849616058189,"{""type"": ""FeatureCollection"", ""features"": [{""i...",0.9999936640027093,-3.0911078359252775,False
2,3264093,2024-07-20 07:05:18,"MULTIPOLYGON(((-14.355698 48.053513,-14.355011...",5,1,0.9159122705459595,76377.44526714,22870602.19789487,213312.21003821932,0101000020E6100000AE579E00649A2DC0A7624577230D...,0.006316202329465258,0.45344901672487353,"{""type"": ""FeatureCollection"", ""features"": [{""i...",0.9999960041891336,-4.458064854763842,False
3,3030492,2024-08-10 11:50:22,"MULTIPOLYGON(((-91.665857 0.583594,-91.66105 0...",5,1,0.9863162040710449,20201.06759414,34918265.7979765,151266.87753361906,0101000020E610000048A41BD082ED56C0F75646F00C07...,0.01917674291267693,0.19226207056489528,"{""type"": ""FeatureCollection"", ""features"": [{""i...",0.40790214755351906,-1.0409978140947282,False
4,3022415,2024-08-21 00:27:07,"MULTIPOLYGON(((-91.35618 0.231345,-91.35412 0....",5,1,0.9152988195419312,11577.03241622,16737114.913490694,84535.91637279905,0101000020E610000032DFBEFF6BD756C028093ABC1562...,0.02943119745078223,0.36656557433898673,"{""type"": ""FeatureCollection"", ""features"": [{""i...",0.3093327878017077,NULL,False


In [ ]:
hitl_gdf.columns

#make hitl class a new column



Index(['id', 'slick_timestamp', 'st_astext', 'machine_cls', 'hitl_cls',
       'machine_confidence', 'length', 'area', 'perimeter', 'centroid',
       'polsby_popper', 'fill_factor', 'centerlines', 'aspect_ratio_factor',
       'max_source_collated_score'],
      dtype='object')

In [ ]:

hitl_gdf_reduced = (hitl_gdf.drop(columns=['hitl_cls','machine_cls','centroid','centerlines', 'st_astext', 'max_source_collated_score', 'slick_timestamp', 'machine_confidence'], axis=1))

In [40]:
hitl_gdf_reduced

,id,length,area,perimeter,polsby_popper,fill_factor,aspect_ratio_factor,is_slick
0,3273057,90479.42156099,10430194.534472644,179240.77044128766,0.004079704134171181,0.17341434501669312,0.9998600689270918,False
1,3278751,187059.15992051,17981273.724743545,347689.96294476854,0.001869157978317394,0.08627849616058189,0.9999936640027093,False
2,3264093,76377.44526714,22870602.19789487,213312.21003821932,0.006316202329465258,0.45344901672487353,0.9999960041891336,False
3,3030492,20201.06759414,34918265.7979765,151266.87753361906,0.01917674291267693,0.19226207056489528,0.40790214755351906,False
4,3022415,11577.03241622,16737114.913490694,84535.91637279905,0.02943119745078223,0.36656557433898673,0.3093327878017077,False
...,...,...,...,...,...,...,...,...
7964,3711339,8567.4194362,1813115.1084502637,21918.181074898355,0.04742706177694859,0.14197932079736814,0.5290733032504711,False
7965,3695903,16576.65736051,996556.0341318846,24299.168876423963,0.021209417859542463,0.015228287762947532,0.8554685236112738,False
7968,3277245,13828.70381106,2345405.0369896293,37708.364498245755,0.020727757767558776,0.1217709868791142,0.9512273807685936,True
7969,3506646,5489.98343405,3303893.486179292,16628.510751621943,0.15015133247697116,0.5593938214986587,0.3746504445126545,True


In [41]:
hitl_gdf_reduced.to_csv('/Users/brendanjarrell/Desktop/ethanHasHisInitialDS.csv', index=False)